# Workshop 11 — Evals: Test Your Agent Like Software

One new idea: a hand-rolled **regression eval harness**. A golden set of cases, each pairing an input with *structural* assertions on the Workshop 9 six-key contract, run through the real `ChatSession`, so a regression fails red **before a student hits it**.

Workshop 9 built the assertable surface. Workshop 10 recorded the evidence. Workshop 11 turns both into a gate.

**Two rules that make evals work:**
- **Assert on decisions, not diction.** `status`, `category`, `missing`/`items` shape come from constrained enums and structure, so they're stable across runs. Prose changes every run, so the only text check allowed is a loose case-insensitive substring. *Asserting on exact prose is the pitfall* — it fails tomorrow on an equally correct paraphrase, and an ignored red suite is worse than none.
- **Never hardcode clock-dependent truth.** "5 days until Thursday" is wrong tomorrow. Compute expected day counts in the harness, at run time.

Nemotron is non-deterministic (temp 0 on a hosted endpoint isn't a fix — batching makes it non-deterministic anyway), so each case runs once and **retries once on failure**: first-try pass = PASS, pass-on-retry = FLAKY, fail twice = FAIL.

Same model, same `ChatSession` from Workshop 10, carried inline so this notebook stands alone.

## Step 1 — Carry the Workshop 10 agent (setup, tools, tracer, session)

In [ ]:
%pip install -q openai numpy

import os, getpass, json, time, uuid
from datetime import datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo
from openai import OpenAI
import numpy as np

if not os.getenv('NVIDIA_API_KEY'):
    os.environ['NVIDIA_API_KEY'] = getpass.getpass('Paste your NVIDIA API key (starts with nvapi-): ')

client = OpenAI(base_url='https://integrate.api.nvidia.com/v1', api_key=os.environ['NVIDIA_API_KEY'])

MODEL = 'nvidia/llama-3.3-nemotron-super-49b-v1.5'
EMBED_MODEL = 'nvidia/nv-embedqa-e5-v5'
LOCAL_TZ = 'America/Los_Angeles'
TRACE_PATH = Path('traces/campus_assistant.jsonl')   # traces hold user data — never commit

knowledge_base = [
    {"title": "USC AI Club meeting",
     "text": "The USC AI Club meets every Thursday at 5 PM in the engineering building, room 204."},
    {"title": "USC GPU lab hours",
     "text": "The USC GPU computing lab is open Monday to Friday from 10 AM to 6 PM."},
    {"title": "NVIDIA Developer Program",
     "text": "USC students can join the NVIDIA Developer Program for free."},
    {"title": "Next USC workshop",
     "text": "The next USC AI Club workshop will cover Retrieval Augmented Generation (RAG)."},
    {"title": "USC AI/ML office hours",
     "text": "Office hours for the USC AI/ML faculty are Tuesdays 2-4 PM."},
    {"title": "USC robotics lab",
     "text": "The USC robotics lab requires safety training before students can use the soldering station."},
    {"title": "USC tutoring",
     "text": "Peer tutoring for introductory Python at USC is available Wednesdays from 1 PM to 3 PM."},
]


def embed_texts(texts, input_type="passage"):
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts,
        extra_body={"input_type": input_type},
    )
    return [np.array(item.embedding, dtype=np.float32) for item in response.data]


def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def retrieve_context(question: str, k: int = 3) -> str:
    q_emb = embed_texts([question], input_type="query")[0]
    scored = [(cosine_similarity(q_emb, item["embedding"]), item) for item in knowledge_base]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return "\n".join(f"- {item['text']}" for _, item in scored[:k])


print("Embedding USC knowledge base...")
for item, emb in zip(knowledge_base, embed_texts([i["text"] for i in knowledge_base], "passage")):
    item["embedding"] = emb
print(f"Embedded {len(knowledge_base)} chunks. Model: {MODEL}")

In [ ]:
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]


def get_current_time(timezone: str = LOCAL_TZ) -> str:
    try:
        zone = ZoneInfo(timezone)
    except Exception:
        zone = ZoneInfo("UTC")
    return datetime.now(zone).strftime("%A, %B %d, %Y at %I:%M %p %Z")


def search_campus_info(query: str) -> str:
    return retrieve_context(query, k=3)


def days_until_weekday(weekday: str) -> str:
    target = weekday.strip().capitalize()
    if target not in WEEKDAYS:
        return f"'{weekday}' is not a valid weekday. Use one of: {', '.join(WEEKDAYS)}."
    today = datetime.now(ZoneInfo(LOCAL_TZ))
    delta = (WEEKDAYS.index(target) - today.weekday()) % 7
    date_str = (today + timedelta(days=delta)).strftime("%B %d, %Y")
    if delta == 0:
        return f"Today is {target} ({date_str}) — that is 0 days away (it's today)."
    return f"The next {target} is in {delta} day(s), on {date_str}. Today is {today.strftime('%A')}."


tools = [
    {
        "type": "function",
        "function": {
            "name": "search_campus_info",
            "description": (
                "Search the USC campus knowledge base for facts about USC clubs "
                "(including the AI Club), labs (GPU lab, robotics lab), workshops, "
                "faculty office hours, peer tutoring, and the NVIDIA Developer "
                "Program. Use this to find WHEN or WHERE something happens. Always "
                "call this for any USC-specific fact — do not answer from your own "
                "knowledge."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string",
                              "description": "The USC campus question or search phrase."},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": (
                "Get the current date, day of week, and time in an IANA time zone. "
                "Use this when the question depends on what day or time it is right now."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {"type": "string",
                                 "description": "IANA time zone, e.g. America/Los_Angeles or UTC."},
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "days_until_weekday",
            "description": (
                "Calculate how many days from today until the next occurrence of a "
                "given weekday (e.g. 'Thursday'). Use this AFTER you know which day "
                "an event happens, to work out how far away it is."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "weekday": {"type": "string",
                                "description": "A weekday name, e.g. Monday, Thursday, Sunday."},
                },
                "required": ["weekday"],
            },
        },
    },
]

available_tools = {
    "search_campus_info": search_campus_info,
    "get_current_time": get_current_time,
    "days_until_weekday": days_until_weekday,
}


def run_tool(name: str, arguments: dict) -> str:
    if name not in available_tools:
        return f"Tool '{name}' is not available."
    try:
        return available_tools[name](**arguments)
    except Exception as exc:
        return f"Tool '{name}' failed: {exc}"

STATUSES = {"answered", "not_found", "needs_clarification"}
CATEGORIES = {"campus_event", "campus_hours", "campus_resource", "comparison", "refusal"}
REQUIRED_KEYS = ("status", "answer", "category", "items", "missing", "sources")

SYSTEM_PROMPT = (
    "/no_think\n\nYou are a USC campus assistant having an ongoing conversation with a student. "
    "You remember everything said earlier in this conversation.\n\n"
    "When a question refers back to something already discussed — words like 'that', "
    "'those', 'then', 'it', or 'the second one' — resolve the reference from the "
    "conversation so far before doing anything else.\n\n"
    "You have three tools: search_campus_info (find USC facts), get_current_time "
    "(today's date and time), and days_until_weekday (days from today until a weekday). "
    "Work step by step: decide what you still need, call ONE tool, read the result, then "
    "continue. Before calling a tool, check whether the conversation ALREADY contains the "
    "fact you need — do not re-search for something you found a turn ago. To compare how "
    "soon two days are, call days_until_weekday for EACH day and compare the numbers it "
    "returns — never estimate the number of days yourself.\n\n"
    "FINAL ANSWER FORMAT. When you are done using tools, your final reply MUST be a single "
    "JSON object and NOTHING else — no prose before or after, no code fences. Use exactly "
    "these keys:\n"
    '  "status": one of "answered", "not_found", or "needs_clarification"\n'
    '  "answer": a one or two sentence plain-language answer for a human\n'
    '  "category": one of "campus_event", "campus_hours", "campus_resource", "comparison", "refusal"\n'
    '  "items": a list of objects holding the specific facts (name, day, time, location, days_until as relevant); may be empty\n'
    '  "missing": a list of things the user asked for that you could not find; may be empty\n'
    '  "sources": a list of the exact knowledge-base lines you used; may be empty\n'
    "Base everything strictly on tool results and the conversation. If you cannot find the "
    'answer, use status "not_found", category "refusal", put what is missing in "missing", '
    'and set "answer" to: I don\'t have that information — check with the USC AI Club.'
)

MAX_STEPS = 5


def parse_json_object(text: str) -> dict:
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("no JSON object found in model output")
    return json.loads(text[start:end + 1])


def validate_answer(data) -> list:
    if not isinstance(data, dict):
        return ["response is not a JSON object"]
    errors = []
    for key in REQUIRED_KEYS:
        if key not in data:
            errors.append(f"missing required key: {key}")
    if data.get("status") not in STATUSES:
        errors.append(f"status must be one of {sorted(STATUSES)}")
    if data.get("category") not in CATEGORIES:
        errors.append(f"category must be one of {sorted(CATEGORIES)}")
    if "answer" in data and not isinstance(data["answer"], str):
        errors.append("answer must be a string")
    for key in ("items", "missing", "sources"):
        if key in data and not isinstance(data[key], list):
            errors.append(f"{key} must be a list")
    if isinstance(data.get("items"), list) and not all(isinstance(it, dict) for it in data["items"]):
        errors.append("each entry in items must be an object")
    return errors


def repair_answer_json(raw_text: str, errors: list) -> dict | None:
    try:
        fix = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            max_tokens=800,
            messages=[
                {"role": "system", "content": (
                    "/no_think\n\nYou fix malformed JSON. Return ONLY a single valid JSON object — "
                    "no prose, no code fences.")},
                {"role": "user", "content": (
                    f"This response was supposed to match a schema but had these problems: "
                    f"{errors}.\n\nRequired keys: status (one of {sorted(STATUSES)}), answer "
                    f"(string), category (one of {sorted(CATEGORIES)}), items (list), missing "
                    f"(list), sources (list).\n\nPreserve all facts. Here is the original:\n"
                    f"{raw_text}\n\nReturn the corrected JSON object only.")},
            ],
        )
        data = parse_json_object(fix.choices[0].message.content or "")
    except Exception:
        return None
    return data if not validate_answer(data) else None


def format_error(missing: str = "valid_json", *, answer: str | None = None) -> dict:
    return {
        "status": "needs_clarification",
        "answer": answer or "I couldn't produce a valid structured response. Please ask again.",
        "category": "refusal",
        "items": [],
        "missing": [missing],
        "sources": [],
    }

In [ ]:
class JsonlTracer:
    """Writes one JSON line per completed turn. Plain file I/O — no framework.

    The privacy rule: a trace holds the user's message, tool results, and the
    final answer — in a real deployment that's personal data. Never log API
    keys, request headers, or the whole self.messages array. And keep the
    trace directory out of git (see .gitignore).
    """

    SCHEMA_VERSION = "ws10.turn.v1"

    def __init__(self, path: Path = TRACE_PATH):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.trace_id = uuid.uuid4().hex[:12]     # one id per session
        self.turn_id = 0
        self._turn = None                          # the record being built

    def begin_turn(self, user_message: str, mode: str) -> None:
        if self._turn is not None:          # a prior turn never closed — keep the evidence
            self.abort_turn("turn abandoned without end_turn")
        self.turn_id += 1
        self._turn = {
            "schema_version": self.SCHEMA_VERSION,
            "trace_id": self.trace_id,
            "turn_id": self.turn_id,
            "timestamp": datetime.now(ZoneInfo("UTC")).isoformat(),
            "model": MODEL,
            "mode": mode,                          # "chat" or "stream"
            "user_message": user_message,
            "steps": [],
            "final": None,
            "total_latency_ms": None,
        }
        self._start = time.perf_counter()

    def record_model_call(self, step: int, latency_ms: int, usage, tool_calls) -> None:
        self._turn["steps"].append({
            "type": "model_call",
            "step": step,
            "latency_ms": latency_ms,
            # usage is best-effort: null when the endpoint didn't report it.
            "usage": usage,
            "tool_calls": [
                {"id": tc["id"], "name": tc["name"], "arguments": tc["arguments"]}
                for tc in tool_calls
            ],
        })

    def record_tool_call(self, step: int, tool_call_id: str, name: str,
                         arguments: dict, result: str, latency_ms: int) -> None:
        self._turn["steps"].append({
            "type": "tool_call",
            "step": step,
            "tool_call_id": tool_call_id,
            "name": name,
            "arguments": arguments,
            "result": result,
            "latency_ms": latency_ms,
        })

    def record_validation(self, parse_ok: bool, errors: list,
                          repair_attempted: bool, repair_succeeded: bool) -> None:
        self._turn["steps"].append({
            "type": "validation",
            "parse_ok": parse_ok,
            "errors": errors,
            "repair_attempted": repair_attempted,
            "repair_succeeded": repair_succeeded,
        })

    def end_turn(self, final: dict) -> None:
        self._turn["final"] = final
        self._turn["total_latency_ms"] = int((time.perf_counter() - self._start) * 1000)
        with self.path.open("a") as f:
            f.write(json.dumps(self._turn) + "\n")
        self._turn = None

    def abort_turn(self, error: str) -> None:
        """A crash mid-turn still leaves a trace — that's when you need one most."""
        if self._turn is None:
            return
        self._turn["error"] = error
        self.end_turn(final=None)

In [ ]:
class ChatSession:
    """Workshop 9's session, now traced. The loop is unchanged — the tracer is
    bolted onto its five seams: turn start, each model call, each tool call,
    validation, and the final answer."""

    def __init__(self, max_turns: int = 8, verbose: bool = True,
                 trace_path: Path = TRACE_PATH):
        self.system = {"role": "system", "content": SYSTEM_PROMPT}
        self.messages = [self.system]
        self.max_turns = max_turns
        self.verbose = verbose
        self.tracer = JsonlTracer(trace_path)      # ← new in Workshop 10

    def reset(self) -> None:
        self.messages = [self.system]

    def _trim(self) -> None:
        user_indices = [i for i, m in enumerate(self.messages) if m.get("role") == "user"]
        if len(user_indices) <= self.max_turns:
            return
        cut = user_indices[-self.max_turns]
        self.messages = [self.system] + self.messages[cut:]

    def _complete(self, stream: bool):
        """One model call. New in Workshop 10: also returns best-effort usage
        (a dict with prompt/completion/total tokens, or None when the endpoint
        didn't report it — never fake zeros)."""
        if not stream:
            resp = client.chat.completions.create(
                model=MODEL, messages=self.messages, tools=tools,
                tool_choice="auto", temperature=0.2, max_tokens=800,
            )
            msg = resp.choices[0].message
            tool_calls = [{"id": tc.id, "name": tc.function.name,
                           "arguments": tc.function.arguments or "{}"}
                          for tc in (msg.tool_calls or [])]
            usage = None
            if getattr(resp, "usage", None):
                usage = {"prompt_tokens": resp.usage.prompt_tokens,
                         "completion_tokens": resp.usage.completion_tokens,
                         "total_tokens": resp.usage.total_tokens}
            return (msg.content or ""), tool_calls, msg.model_dump(exclude_none=True), usage

        stream_resp = client.chat.completions.create(
            model=MODEL, messages=self.messages, tools=tools,
            tool_choice="auto", temperature=0.2, max_tokens=800, stream=True,
        )
        text_parts, fragments = [], {}
        for chunk in stream_resp:
            if not chunk.choices:
                continue
            delta = chunk.choices[0].delta
            if delta.content:
                text_parts.append(delta.content)
            for tc in (delta.tool_calls or []):
                slot = fragments.setdefault(tc.index, {"id": "", "name": "", "arguments": ""})
                if tc.id and not slot["id"]:
                    slot["id"] = tc.id
                if tc.function and tc.function.name and not slot["name"]:
                    slot["name"] = tc.function.name
                if tc.function and tc.function.arguments:
                    slot["arguments"] += tc.function.arguments
        text = "".join(text_parts)
        tool_calls = [fragments[i] for i in sorted(fragments)]
        assistant_msg = {"role": "assistant"}
        if tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc["id"], "type": "function",
                 "function": {"name": tc["name"], "arguments": tc["arguments"]}}
                for tc in tool_calls
            ]
            if text:
                assistant_msg["content"] = text
        else:
            assistant_msg["content"] = text
        # Streamed responses usually don't carry usage — log null, don't guess.
        return text, tool_calls, assistant_msg, None

    def _finalize_json(self, raw_text: str):
        """New in Workshop 10: returns (data, validation_meta) so the tracer can
        record what happened without tangling the parsing logic."""
        meta = {"parse_ok": True, "errors": [], "repair_attempted": False,
                "repair_succeeded": False}
        if not raw_text.strip():
            # An empty reply is a real llama failure mode. Don't send it to repair —
            # with nothing to preserve, the repair model would invent facts.
            meta["parse_ok"] = False
            meta["errors"] = ["empty model response"]
            return format_error("empty_model_response"), meta
        try:
            data = parse_json_object(raw_text)
            meta["errors"] = validate_answer(data)
        except (ValueError, json.JSONDecodeError):
            data, meta["parse_ok"] = None, False
            meta["errors"] = ["response was not valid JSON"]
        if meta["errors"]:
            if self.verbose:
                print(f"  (final JSON invalid: {meta['errors']} — attempting one repair)")
            meta["repair_attempted"] = True
            repaired = repair_answer_json(raw_text, meta["errors"])
            meta["repair_succeeded"] = repaired is not None
            data = repaired if repaired is not None else format_error()
        return data, meta

    def _run_turn(self, user_message: str, stream: bool) -> dict:
        checkpoint = len(self.messages)     # so a failed turn can be rolled back
        self.messages.append({"role": "user", "content": user_message})
        self.tracer.begin_turn(user_message, mode="stream" if stream else "chat")
        try:
            return self._run_turn_traced(stream)
        except BaseException as exc:        # BaseException: Ctrl-C must leave a trace too
            # Write the partial trace before the exception propagates — a crash
            # mid-turn is exactly the moment you'll want the record. Then roll the
            # history back to the checkpoint so the session stays usable: a half
            # turn (assistant tool_calls with no tool replies) breaks later calls.
            self.tracer.abort_turn(f"{type(exc).__name__}: {exc}")
            del self.messages[checkpoint:]
            raise

    def _run_turn_traced(self, stream: bool) -> dict:
        for step in range(1, MAX_STEPS + 1):
            t0 = time.perf_counter()
            text, tool_calls, assistant_msg, usage = self._complete(stream)
            self.tracer.record_model_call(
                step, int((time.perf_counter() - t0) * 1000), usage, tool_calls)

            if tool_calls:
                self.messages.append(assistant_msg)
                for tc in tool_calls:
                    try:
                        arguments = json.loads(tc["arguments"] or "{}")
                    except json.JSONDecodeError:
                        arguments = {}
                    t1 = time.perf_counter()
                    result = run_tool(tc["name"], arguments)
                    self.tracer.record_tool_call(
                        step, tc["id"], tc["name"], arguments, str(result),
                        int((time.perf_counter() - t1) * 1000))
                    if self.verbose:
                        print(f"  step {step} · {tc['name']}({json.dumps(arguments)}) -> {result}")
                    self.messages.append({"role": "tool", "tool_call_id": tc["id"],
                                          "name": tc["name"], "content": str(result)})
                continue

            data, vmeta = self._finalize_json(text)
            self.tracer.record_validation(vmeta["parse_ok"], vmeta["errors"],
                                          vmeta["repair_attempted"], vmeta["repair_succeeded"])
            self.messages.append({"role": "assistant", "content": json.dumps(data)})
            self._trim()
            self.tracer.end_turn(data)
            return data

        data = format_error(
            "step_limit_reached",
            answer="I reached the step limit before finishing — try asking a narrower question.",
        )
        # Keep every trace line the same shape: this turn gets a validation event too.
        self.tracer.record_validation(parse_ok=False, errors=["step_limit_reached"],
                                      repair_attempted=False, repair_succeeded=False)
        self.messages.append({"role": "assistant", "content": json.dumps(data)})
        self._trim()
        self.tracer.end_turn(data)
        return data

    def chat(self, user_message: str) -> dict:
        return self._run_turn(user_message, stream=False)

    def stream(self, user_message: str) -> dict:
        return self._run_turn(user_message, stream=True)

## Step 2 — The assertion vocabulary

`check_case` runs every checker named in a case's `expect`, collects **all** failures, always runs `validate_answer` first, guards against a crashed (non-dict) turn, and **raises on an unknown `expect` key** so a typo'd assertion can't silently pass. Tool-path checks read the trace this run just wrote.

In [ ]:
EVAL_TRACE = Path("traces/evals.jsonl")   # separate from the chat trace

def days_until(weekday):
    """Same arithmetic the tool uses, so expected values track the calendar."""
    today = datetime.now(ZoneInfo(LOCAL_TZ))
    return (WEEKDAYS.index(weekday) - today.weekday()) % 7

def case_steps(path):
    # all steps across this case's turns; each attempt clears the file first,
    # so everything here belongs to the current run (also catches earlier turns)
    steps = []
    for line in (Path(path).read_text().splitlines() if Path(path).exists() else []):
        steps.extend(json.loads(line).get("steps", []))
    return steps

def _tool_calls(steps):
    return [(s.get("name"), s.get("arguments", {}))
            for s in steps if s.get("type") == "tool_call"]

def _lc(x): return str(x).lower() if x is not None else ""
def _lst(f, key): v = f.get(key); return v if isinstance(v, list) else []
def _int(x):
    import re; m = re.search(r"-?\d+", str(x)); return int(m.group()) if m else None

CHECKERS = {
    "status": lambda f,s,e: None if f.get("status")==e else f"status: expected {e!r}, got {f.get('status')!r}",
    "category": lambda f,s,e: None if f.get("category")==e else f"category: expected {e!r}, got {f.get('category')!r}",
    "category_in": lambda f,s,e: None if f.get("category") in e else f"category: expected one of {e}, got {f.get('category')!r}",
    "min_items": lambda f,s,e: None if len(_lst(f,"items"))>=e else f"min_items: expected >= {e}, got {len(_lst(f,'items'))}",
    "items_empty": lambda f,s,e: None if (len(_lst(f,"items"))==0)==e else f"items_empty: expected {e}",
    "missing_empty": lambda f,s,e: None if (len(_lst(f,"missing"))==0)==e else f"missing_empty: expected {e}, missing={f.get('missing')}",
    "missing_nonempty": lambda f,s,e: None if (len(_lst(f,"missing"))>0)==e else f"missing_nonempty: expected {e}, missing={f.get('missing')}",
    "sources_nonempty": lambda f,s,e: None if (len(_lst(f,"sources"))>0)==e else f"sources_nonempty: expected {e}",
    "answer_contains_any": lambda f,s,e: None if any(_lc(x) in _lc(f.get("answer")) for x in e) else f"answer_contains_any: none of {e} in answer",
    "answer_not_contains": lambda f,s,e: None if not any(_lc(x) in _lc(f.get("answer")) for x in e) else f"answer_not_contains: found one of {e}",
    "item_has_keys": lambda f,s,e: None if any(all(k in it for k in e) for it in _lst(f,"items") if isinstance(it,dict)) else f"item_has_keys: no items entry has all of {e}",
    "item_field_equals": lambda f,s,e: None if all(any(isinstance(it,dict) and _int(it.get(fld)) is not None and _int(it.get(fld))==_int(v) for it in _lst(f,"items")) for fld,vals in e.items() for v in (vals if isinstance(vals,list) else [vals])) else f"item_field_equals: no item satisfies {e}",
    "tools_called": lambda f,s,e: None if set(e)<={n for n,_ in _tool_calls(s)} else f"tools_called: expected {e}, saw {sorted({n for n,_ in _tool_calls(s)})}",
    "tool_args_contain": lambda f,s,e: None if all(any(n==t and _lc(sub) in _lc(json.dumps(a)) for n,a in _tool_calls(s)) for t,sub in e.items()) else f"tool_args_contain: expected {e}",
}

def check_case(final, steps, expect):
    unknown = set(expect) - set(CHECKERS)
    if unknown:
        raise ValueError(f"unknown expect key(s): {sorted(unknown)}")
    if not isinstance(final, dict):          # a crashed/empty turn is a failure, not a traceback
        return [f"final is not a dict (got {type(final).__name__})"]
    failures = []
    errs = validate_answer(final)
    if errs:
        failures.append(f"schema invalid: {errs}")
    for key, expected in expect.items():
        msg = CHECKERS[key](final, steps, expected)
        if msg:
            failures.append(msg)
    return failures

## Step 3 — The golden set (grounded in the real knowledge base, built at run time)

Clock-dependent expectations (the comparison winner, the days-until count) are computed **inside `build_cases()`** at run time, so the suite stays correct tomorrow.

In [ ]:
# Built at run time so clock-dependent expectations never go stale.
def build_cases():
    return [
        {"name": "ai_club_answered", "regression": False,
         "turns": ["When does the USC AI Club meet?"],
         "expect": {"status": "answered", "category": "campus_event", "min_items": 1,
                    "missing_empty": True, "sources_nonempty": True,
                    "answer_contains_any": ["thursday"], "tools_called": ["search_campus_info"]}},
        {"name": "gpu_lab_hours_answered", "regression": False,
         "turns": ["What are the USC GPU lab hours?"],
         "expect": {"status": "answered", "category": "campus_hours", "min_items": 1,
                    "sources_nonempty": True, "answer_contains_any": ["10", "6", "monday", "friday"]}},
        {"name": "wifi_stays_refusal", "regression": True,   # broke during the Nemotron migration
         "turns": ["What is the campus wifi password?"],
         "expect": {"status": "not_found", "category": "refusal", "items_empty": True,
                    "missing_nonempty": True, "answer_contains_any": ["check with the usc ai club"],
                    "answer_not_contains": ["the password is", "password:"]}},
        {"name": "sooner_comparison", "regression": False,
         "turns": ["Which is sooner, the AI Club meeting or the AI/ML office hours?"],
         "expect": {"status": "answered", "category": "comparison", "min_items": 2,
                    "tools_called": ["days_until_weekday"],
                    "item_field_equals": {"days_until": [days_until("Thursday"), days_until("Tuesday")]}}},
        {"name": "memory_days_until", "regression": False,   # multi-turn: "that" must resolve
         "turns": ["When does the USC AI Club meet?", "How many days until that?"],
         "expect": {"status": "answered", "category_in": ["campus_event", "campus_hours", "comparison"],
                    "tool_args_contain": {"days_until_weekday": "thursday"},
                    "item_field_equals": {"days_until": days_until("Thursday")}}},
    ]

## Step 4 — Run the suite

Each case runs in a fresh `ChatSession` (no memory bleed), clears the trace first (so assertions read only this run), retries once on failure, and grades the **last** turn. A network blip is caught and recorded as a failure instead of aborting the suite.

In [ ]:
import time

def run_case(case):
    def attempt():
        if EVAL_TRACE.exists():
            EVAL_TRACE.unlink()          # clean slate: reader sees only this run's turns
        session = ChatSession(verbose=False, trace_path=EVAL_TRACE)   # fresh: no memory bleed
        final = None
        try:
            for turn in case["turns"]:
                final = session.chat(turn)
        except Exception as exc:         # a network/API blip shouldn't abort the whole suite
            return [f"agent raised: {type(exc).__name__}: {exc}"]
        return check_case(final, case_steps(EVAL_TRACE), case["expect"])
    failures = attempt()
    if not failures:
        return "PASS", []
    retry = attempt()
    return ("FLAKY", failures) if not retry else ("FAIL", retry)

print("── Workshop 11: eval suite ──  (temp=0.2, run-once + retry-on-fail)")
passed = flaky = failed = 0
for case in build_cases():
    t0 = time.perf_counter()
    verdict, failures = run_case(case)
    tag = "REGRESSION " if case.get("regression") else ""
    print(f"{verdict:5s}  {tag}{case['name']:28s} {time.perf_counter()-t0:5.1f}s")
    if verdict == "PASS":
        passed += 1
    elif verdict == "FLAKY":
        flaky += 1
        print("         (passed only on retry — investigate the prompt)")
    else:
        failed += 1
        for msg in failures:
            print(f"         - {msg}")
print(f"\neval summary: {passed} passed, {flaky} flaky, {failed} failed")

## What just happened

You ran your agent through a **golden set** and got a red/green verdict per case — Workshop 9's contract and Workshop 10's traces, cashed in. The suite asserts on *decisions* (status, category, tool path), never prose, so a correct reword stays green while a wrong decision goes red.

- The **wifi regression** case pins the exact failure that broke during the Nemotron migration, plus a leak guard.
- Clock-dependent truth is **computed at run time**, so the suite doesn't rot overnight.
- **PASS / FLAKY / FAIL** gives a stochastic system honest semantics a plain pass/fail can't.
- A crashed turn becomes a recorded failure, not a traceback that kills the run.

### Why not pytest / DeepEval / NeMo Evaluator?

- **pytest** — the cases port to `@pytest.mark.parametrize` almost mechanically (a good exercise), but binary pass/fail has no room for FLAKY, and discovery fights the data-driven `for case in build_cases()` loop. The ~60-line runner shows there's no magic.
- **DeepEval / Ragas** — add LLM-as-judge metrics; a judge is a second non-deterministic model grading the first. The six-key contract lets us assert deterministically, so we don't pay that cost yet.
- **NVIDIA NeMo Evaluator** — the production graduation path: custom datasets and benchmarks at scale against NIM endpoints, judge models, results tracked over time. This harness is a working miniature — swap the runner for NeMo Evaluator, keep the golden cases, stay in the NVIDIA ecosystem.

## The full series

- **1** First call · **2** RAG · **3** Guardrails · **4** NIM on your GPU · **5** Tool calling · **6** ReAct · **7** Memory · **8** Streaming · **9** Structured outputs · **10** Traces
- **Workshop 11 (this notebook):** Evals — test the agent like software.

Star and fork the repo: https://github.com/torkian/nvidia-nim-workshop